### Dreieckspuzzle – Lösung Schritt für Schritt

*39. Bundeswettbewerb Informatik (2020), 1. Runde, Aufgabe 2*

Lizzy hat ein Dreieckspuzzle geschenkt bekommen. Es besteht aus neun dreieckigen Teilen. Auf den einzelnen Teilen sind
Figuren abgebildet, aber immer nur zur Hälfte.

Die Teile sollen zu einem großen Dreieck zusammengesetzt werden, und zwar so, dass die Figurenhälften zueinander passen.
Lizzy lernt gerade programmieren und denkt sich: „Bestimmt kann das Puzzle auch mit einem Computerprogramm gelöst werden.
Und das Programm schreibe ich gleich so, dass es auch andere Dreieckspuzzles mit neun Teilen lösen kann: mit mehr
verschiedenen und anders angeordneten Figuren.“

#### Aufgabe 2

Schreibe ein Programm, das Dreieckspuzzles lösen kann. Es soll zunächst die Beschreibung eines Puzzles einlesen. Für das
eingelesene Puzzle soll es entscheiden, ob es eine Lösung gibt und, falls ja, eine Lösung ausgeben.
Wende dein Programm mindestens auf die Puzzlebeschreibungen an, die du auf den BWINF-Webseiten findest (`puzzle0.txt` bis `puzzle3.txt`), und dokumentiere die Ergebnisse.

#### Beispieldaten

- 1. Zeile: Anzahl m der Figuren, 2. Zeile: Anzahl n der Teile (9)
- je Teil eine Zeile mit drei Zahlen für die drei Seiten: eine positive Zahl `z` steht für die Oberseite der Figur `z`, eine negative Zahl `-z` für die Unterseite.
  Zwei Seiten passen also zusammen, wenn ihre Zahlen **zusammen 0** ergeben.

`puzzle0.txt` (Anzahl Figuren 3, Anzahl Teile 9, dann die neun Teile):

```
3
9
-1 -2 1
...
```

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Backtracking:** Teile nacheinander auf die Positionen 0, 1, 2, ... legen. Sobald an einer Kante zwei Hälften nicht zusammenpassen, ist der Zweig erledigt, und wir probieren etwas anderes.
  Das spart gegenüber dem Durchprobieren aller Anordnungen fast alles (es gäbe 9! · 3^9 = 7 142 567 040).
- **Rekursion:** Das Legen des nächsten Teils ist dieselbe Aufgabe wie das Legen des vorigen, nur mit einem Teil weniger übrig.
- **Modulo zum Drehen:** Ein Teil dreht man, indem man mit `% 3` zyklisch weiterzählt: nach Seite 2 kommt wieder Seite 0.
- **Geeignete Darstellung:** Die Nachbarschaften der Positionen schreibt man einmal als Liste von Kanten auf. Dann ist das Prüfen einer Anordnung eine einfache Schleife.

**Nützliche Funktionen**
- `open(datei, encoding='utf-8-sig')`, `zeile.split()`, `int(...)` zum Einlesen.
- `list(anordnung)` kopiert eine Liste (wichtig, wenn man eine Lösung später noch braucht).

## Die Idee

**Teile.** Jedes Teil ist eine Liste aus drei Zahlen, die der Reihe nach im Uhrzeigersinn die drei Seiten beschreiben. Drehen wir das Teil, werden die Zahlen **zyklisch verschoben**:
`[1, 2, 3]` wird zu `[2, 3, 1]` und dann zu `[3, 1, 2]`. Mehr Drehungen gibt es nicht: Dreimal drehen ergibt wieder das Teil selbst. Wir merken uns statt eines gedrehten Teils
nur die Drehung `d` in 0, 1, 2: Die Zahl an Seite `s` ist dann `teil[(s + d) % 3]`.

*Spiegeln geht nicht, und das müssen wir auch nicht.* Ob die Zahlen in der Datei im oder gegen den Uhrzeigersinn stehen, ist egal: Spiegelt man das ganze Puzzle, bleibt jede Lösung eine Lösung.

**Positionen.** Die neun Positionen nummerieren wir von oben nach unten und in jeder Reihe von links nach rechts. Die Teile mit der Spitze nach oben nennen wir Typ A, die mit der Spitze nach unten Typ B.
Reihe `r` (von 0 an) hat `2r + 1` Positionen und beginnt bei `r * r`.

```
            0                    A: Seite 0 = rechts, 1 = unten, 2 = links
         1  2  3                 B: Seite 0 = oben,   1 = rechts, 2 = links
      4  5  6  7  8              (jeweils im Uhrzeigersinn)
      A  B  A  B  A
```

Innerhalb einer Reihe wechseln sich A und B ab (A, B, A, ...). **Kanten**, an denen zwei Teile aneinanderstoßen, gibt es zwei Arten:

- **waagerecht** zwischen Nachbarn einer Reihe: A und sein rechter Nachbar B (rechte Seite 0 von A, linke Seite 2 von B), und B und sein rechter Nachbar A (Seite 1 von B, Seite 2 von A);
- **senkrecht** unter jedem B: oben (Seite 0) liegt es am Boden (Seite 1) des A-Teils in der Reihe darüber, und das ist die Position `(r-1)*(r-1) + k - 1`, wenn B in Reihe `r` den Index `k` hat.

**Suche.** Wir legen die Positionen der Reihe nach. Für Position `p` probieren wir jedes noch nicht verbrauchte Teil in jeder der drei Drehungen. Passt es zu allen schon liegenden Nachbarn (Zahlen ergeben zusammen 0),
legen wir es und machen mit `p + 1` weiter. Geht es nicht weiter, nehmen wir das Teil wieder weg (Backtracking).

## Schritt 1: Einlesen und Drehen

`lies_puzzle` gibt die Teile als Liste von Listen zurück. `zahl(teil, seite, drehung)` ist die Zahl, die ein um `drehung` gedrehtes Teil an der Seite `seite` hat.

In [ ]:
def lies_puzzle(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        zeilen = f.read().splitlines()
    n = int(zeilen[1])
    teile = []
    for zeile in zeilen[2:2 + n]:
        a, b, c = zeile.split()
        teile.append([int(a), int(b), int(c)])
    return teile


def zahl(teil, seite, drehung):
    return teil[(seite + drehung) % 3]


teile = lies_puzzle('beispieldaten/puzzle0.txt')
print(teile)
print([zahl(teile[0], seite, 0) for seite in range(3)])
print([zahl(teile[0], seite, 1) for seite in range(3)])
print([zahl(teile[0], seite, 2) for seite in range(3)])

[[-1, -2, 1], [2, -1, -1], [-1, -2, 2], [-1, 3, 1], [2, -3, 3], [-1, 3, -2], [2, 2, -1], [-3, 2, -1], [-2, 1, -3]]
[-1, -2, 1]
[-2, 1, -1]
[1, -1, -2]


Das erste Teil von `puzzle0.txt` ist `[-1, -2, 1]`. Gedreht erscheinen dieselben Zahlen in anderer Reihenfolge.

## Schritt 2: Die Kanten

Die Funktion `finde_kanten(n)` liefert für n Positionen (n = 1, 4, 9, 16, ... ist immer eine Quadratzahl, siehe unten) eine Liste von Kanten `(a, seite_a, b, seite_b)`:
Die Seite `seite_a` von Position `a` stößt an Seite `seite_b` von Position `b`. Dabei ist immer `a < b`.
Bei neun Teilen sollten es 9 Kanten sein: 9 Teile haben zusammen 27 Seiten, davon liegen 9 am Rand (3 je Seite des großen Dreiecks), bleiben 18 innere Seiten, also 9 Kanten.

In [ ]:
def finde_kanten(n):
    reihen = int(n ** 0.5 + 0.5)               # n ist eine Quadratzahl
    kanten = []
    for r in range(reihen):
        for k in range(2 * r + 1):             # k: Index in der Reihe, gerade = A, ungerade = B
            p = r * r + k                      # Nummer der Position
            if k < 2 * r:                      # es gibt einen rechten Nachbarn
                if k % 2 == 0:
                    kanten.append((p, 0, p + 1, 2))      # A rechts (0) an B links (2)
                else:
                    kanten.append((p, 1, p + 1, 2))      # B rechts (1) an A links (2)
            if k % 2 == 1:                     # B hat oben einen Nachbarn in der Reihe darüber
                darueber = (r - 1) * (r - 1) + k - 1
                kanten.append((darueber, 1, p, 0))       # A unten (1) an B oben (0)
    return kanten


kanten9 = finde_kanten(9)
print(len(kanten9), 'Kanten')
for kante in kanten9:
    print(kante)

9 Kanten
(1, 0, 2, 2)
(2, 1, 3, 2)
(0, 1, 2, 0)
(4, 0, 5, 2)
(5, 1, 6, 2)
(1, 1, 5, 0)
(6, 0, 7, 2)
(7, 1, 8, 2)
(3, 1, 7, 0)


Zur Kontrolle: Position 0 (oben) stößt mit Seite 1 (unten) an Position 2 (das B darunter) mit Seite 0 (oben). Das sehen wir in der Liste als `(0, 1, 2, 0)`.
Dass es genau neun Kanten sind, passt zur Rechnung oben. Auch eine Nummerierung für 16 oder 25 Teile ergibt sich so: Dann hat man 4 oder 5 Reihen. Die Reihen haben 1, 3, 5, 7, ... Positionen,
und `1 + 3 + 5 + ... + (2r − 1) = r²`, deshalb muss die Zahl der Teile eine Quadratzahl sein.

## Schritt 3: Wann passt ein Teil?

Beim Backtracking legen wir die Positionen in aufsteigender Reihenfolge. Wenn wir Position `p` besetzen, sind nur die Nachbarn mit **kleinerer** Nummer schon da. Für jede Position
halten wir deshalb vorab die Kanten zu früheren Positionen bereit: `(eigene Seite, frühere Position, Seite dort)`. Diese Tabelle bauen wir einmal aus den Kanten.

In [ ]:
def kanten_nach_hinten(n):
    nach_hinten = []
    for p in range(n):
        nach_hinten.append([])
    for a, seite_a, b, seite_b in finde_kanten(n):      # immer a < b
        nach_hinten[b].append((seite_b, a, seite_a))
    return nach_hinten


nach_hinten9 = kanten_nach_hinten(9)
for p in range(9):
    print(p, nach_hinten9[p])

0 []
1 []
2 [(2, 1, 0), (0, 0, 1)]
3 [(2, 2, 1)]
4 []
5 [(2, 4, 0), (0, 1, 1)]
6 [(2, 5, 1)]
7 [(2, 6, 0), (0, 3, 1)]
8 [(2, 7, 1)]


Die Positionen 0, 1 und 4 haben keine früheren Nachbarn (alle ihre Kanten führen zu späteren Positionen). Die A-Teile 3, 6 und 8 haben genau einen, nämlich den linken Nachbarn.
Die B-Teile 2, 5 und 7 haben zwei: den linken Nachbarn und das A-Teil darüber. Position 2 zum Beispiel: `(2, 1, 0)` heißt, ihre Seite 2 (links) liegt an Seite 0 (rechts) von Position 1,
und `(0, 0, 1)` heißt, ihre Seite 0 (oben) liegt an Seite 1 (unten) von Position 0.

Mit dieser Tabelle prüft `passt` ein Teil `t` mit Drehung `d` für Position `p`: Wir gehen alle früheren Nachbarn durch. Für jeden schauen wir nach, welches Teil mit welcher Drehung dort liegt (`anordnung[q]`) und ob die beiden Zahlen zusammen 0 ergeben.

In [ ]:
def passt(teil, drehung, p, teile, anordnung, nach_hinten):
    for eigene_seite, q, seite_q in nach_hinten[p]:
        nummer_q, drehung_q = anordnung[q]
        meine_zahl = zahl(teil, eigene_seite, drehung)
        zahl_q = zahl(teile[nummer_q], seite_q, drehung_q)
        if meine_zahl + zahl_q != 0:
            return False
    return True


# Kleiner Test mit puzzle0: Teil 0 und Teil 1 liegen ungedreht auf Position 0 und 1.
# Passt Teil 2 in einer der drei Drehungen auf Position 2?
teile = lies_puzzle('beispieldaten/puzzle0.txt')
anordnung_test = [(0, 0), (1, 0)] + [None] * 7
for d in range(3):
    print('Teil 2, Drehung', d, ':', passt(teile[2], d, 2, teile, anordnung_test, nach_hinten9))

Teil 2, Drehung 0 : False
Teil 2, Drehung 1 : False
Teil 2, Drehung 2 : True


Nur bei Drehung 2 stimmen beide Kanten. Teil 2 ist `[-1, -2, 2]`. Mit Drehung 2 hat es oben (Seite 0) die 2 und links (Seite 2) die −2.
Darüber liegt Teil 0 `[-1, -2, 1]` mit der −2 an seiner Unterseite (Seite 1): 2 + (−2) = 0. Links liegt Teil 1 `[2, -1, -1]` mit der 2 an seiner rechten Seite (Seite 0): −2 + 2 = 0.

## Schritt 4: Backtracking

`platziere(p, ...)` setzt ab Position `p` fort. Ist `p` gleich der Zahl der Teile, ist das Puzzle voll: Wir merken uns die Anordnung als Lösung (als Kopie mit `list(...)`).
Sonst probieren wir alle noch freien Teile in allen drei Drehungen.

Zwei Betriebsarten: Mit `alle=False` hört die Suche bei der ersten Lösung auf (Rückgabe `True` reicht nach oben durch), mit `alle=True` sucht sie alle Lösungen.
Außerdem zählen wir die Aufrufe von `platziere`, um die Arbeit zu vergleichen.

In [ ]:
def platziere(p, teile, nach_hinten, anordnung, benutzt, loesungen, zaehler, alle):
    zaehler[0] += 1
    if p == len(teile):
        loesungen.append(list(anordnung))
        return not alle                       # True: Suche beenden
    for nummer in range(len(teile)):
        if not benutzt[nummer]:
            for drehung in range(3):
                if passt(teile[nummer], drehung, p, teile, anordnung, nach_hinten):
                    anordnung[p] = (nummer, drehung)
                    benutzt[nummer] = True
                    fertig = platziere(p + 1, teile, nach_hinten, anordnung, benutzt, loesungen, zaehler, alle)
                    benutzt[nummer] = False
                    if fertig:
                        return True
    anordnung[p] = None
    return False


def loese(teile, alle=False):
    n = len(teile)
    nach_hinten = kanten_nach_hinten(n)
    loesungen = []
    zaehler = [0]
    platziere(0, teile, nach_hinten, [None] * n, [False] * n, loesungen, zaehler, alle)
    return loesungen, zaehler[0]

Das Ergebnis ist eine Liste `anordnung`: Für jede Position (0 bis 8) das Paar `(Nummer des Teils, Drehung)`. Die Teile zählen wir ab 0.

## Schritt 5: Ausgabe und Kontrolle

Zum Anzeigen schreiben wir pro Reihe die Positionen mit Teilnummer (ab 1, wie in der Datei: Zeile 3 der Datei ist Teil 1) und den drei Zahlen, wie sie in der Anordnung an den Seiten stehen.
`pruefe_loesung` ist unabhängig vom Suchen: Sie prüft **alle** Kanten und die Tatsache, dass jedes Teil genau einmal vorkommt.

In [ ]:
def zeige(teile, anordnung):
    reihen = int(len(teile) ** 0.5 + 0.5)
    for r in range(reihen):
        stuecke = []
        for k in range(2 * r + 1):
            nummer, drehung = anordnung[r * r + k]
            seiten = [zahl(teile[nummer], s, drehung) for s in range(3)]
            stuecke.append(f'Teil {nummer + 1} (Dreh. {drehung}) {seiten}')
        print('   '.join(stuecke))


def pruefe_loesung(teile, anordnung):
    if sorted(nummer for nummer, drehung in anordnung) != list(range(len(teile))):
        return False
    for a, seite_a, b, seite_b in finde_kanten(len(teile)):
        zahl_a = zahl(teile[anordnung[a][0]], seite_a, anordnung[a][1])
        zahl_b = zahl(teile[anordnung[b][0]], seite_b, anordnung[b][1])
        if zahl_a + zahl_b != 0:
            return False
    return True


for nr in range(4):
    teile = lies_puzzle(f'beispieldaten/puzzle{nr}.txt')
    loesungen, aufrufe = loese(teile)
    print(f'puzzle{nr}: {len(loesungen)} Lösung gefunden, {aufrufe} Aufrufe, geprüft: {pruefe_loesung(teile, loesungen[0])}')
    zeige(teile, loesungen[0])
    print()

puzzle0: 1 Lösung gefunden, 19 Aufrufe, geprüft: True
Teil 1 (Dreh. 0) [-1, -2, 1]
Teil 2 (Dreh. 0) [2, -1, -1]   Teil 3 (Dreh. 2) [2, -1, -2]   Teil 4 (Dreh. 0) [-1, 3, 1]
Teil 7 (Dreh. 0) [2, 2, -1]   Teil 9 (Dreh. 1) [1, -3, -2]   Teil 6 (Dreh. 2) [-2, -1, 3]   Teil 5 (Dreh. 1) [-3, 3, 2]   Teil 8 (Dreh. 1) [2, -1, -3]

puzzle1: 1 Lösung gefunden, 5850 Aufrufe, geprüft: True
Teil 3 (Dreh. 0) [-1, -1, 3]
Teil 2 (Dreh. 0) [2, -3, -1]   Teil 4 (Dreh. 0) [1, -1, -2]   Teil 7 (Dreh. 1) [-2, -3, 1]
Teil 8 (Dreh. 1) [-2, 3, -1]   Teil 9 (Dreh. 0) [3, -2, 2]   Teil 1 (Dreh. 0) [1, -1, 2]   Teil 6 (Dreh. 1) [3, 3, -1]   Teil 5 (Dreh. 1) [3, -1, -3]

puzzle2: 1 Lösung gefunden, 355 Aufrufe, geprüft: True
Teil 1 (Dreh. 1) [-2, -1, -3]
Teil 4 (Dreh. 0) [-3, -1, 1]   Teil 5 (Dreh. 2) [1, 2, 3]   Teil 2 (Dreh. 1) [-4, 1, -2]
Teil 9 (Dreh. 2) [4, -2, -3]   Teil 3 (Dreh. 0) [1, 2, -4]   Teil 6 (Dreh. 2) [-4, -3, -2]   Teil 8 (Dreh. 2) [-1, 3, 4]   Teil 7 (Dreh. 1) [1, -2, -3]

puzzle3: 1 Lösung gef

## Schritt 6: Alle Lösungen, und ein Puzzle ohne Lösung

Wie sicher ist die Antwort „unlösbar“? Die Suche probiert alles, was nicht schon vorher scheitert. Findet sie nichts, gibt es keine Lösung. Das zeigen wir mit einem selbst gebauten Puzzle:
In `puzzle3.txt` ändern wir beim Teil 2 die erste Zahl von 9 auf −9. Dann passt dessen Figurenhälfte zu keiner anderen mehr.

Mit `alle=True` zählen wir außerdem alle Lösungen. Weil man jede Lösung als Ganzes um 120° drehen kann, ohne dass sie ungültig wird, ist die Zahl der Lösungen immer ein **Vielfaches von 3**
(das Dreieck bleibt ein Dreieck, nur liegen die Teile auf anderen Positionen). Wirklich verschieden sind also ein Drittel davon.

In [ ]:
for nr in range(4):
    teile = lies_puzzle(f'beispieldaten/puzzle{nr}.txt')
    alle_loesungen, aufrufe_alle = loese(teile, alle=True)
    _, aufrufe_erste = loese(teile)
    print(f'puzzle{nr}: {len(alle_loesungen)} Lösungen (verschieden bis auf Drehung: {len(alle_loesungen) // 3}), '
          f'Aufrufe bis zur ersten: {aufrufe_erste}, für alle: {aufrufe_alle}')

teile = lies_puzzle('beispieldaten/puzzle3.txt')
teile[1][0] = -9
loesungen, aufrufe = loese(teile)
print('puzzle3 verändert:', len(loesungen), 'Lösungen,', aufrufe, 'Aufrufe')

puzzle0: 84 Lösungen (verschieden bis auf Drehung: 28), Aufrufe bis zur ersten: 19, für alle: 28791
puzzle1: 12 Lösungen (verschieden bis auf Drehung: 4), Aufrufe bis zur ersten: 5850, für alle: 23362
puzzle2: 33 Lösungen (verschieden bis auf Drehung: 11), Aufrufe bis zur ersten: 355, für alle: 12172
puzzle3: 3 Lösungen (verschieden bis auf Drehung: 1), Aufrufe bis zur ersten: 61, für alle: 950
puzzle3 verändert: 0 Lösungen, 888 Aufrufe


**Ergebnis für die Dokumentation:** Alle vier Beispiele sind lösbar. Bis zur ersten Lösung braucht die Suche wenige hundert bis wenige tausend Aufrufe (19, 5850, 355, 61), das passt zu den Lösungshinweisen
(„nur wenige tausend Iterationen“). Das Durchsuchen **aller** Möglichkeiten (nötig, um „unlösbar“ zu beweisen oder alle Lösungen zu zählen) kostet bei diesen Beispielen 950 bis rund 29 000 Aufrufe,
statt der 7 Milliarden Anordnungen ohne frühes Abbrechen. Die gefundene Lösung für `puzzle3.txt` ist (bis auf Drehung des ganzen Dreiecks) die einzige.
Die Lösungsbilder in den Lösungshinweisen sind nur für `puzzle3.txt` abgedruckt; unsere Ausgabe ist die Anordnung mit Teilnummer und Drehung, die sich leicht nachzeichnen lässt.

## Für Neugierige: größere Puzzles

Weil `finde_kanten` für jede Quadratzahl funktioniert, löst dasselbe Programm auch Puzzles mit 16 oder 25 Teilen. Dazu bauen wir uns solche Puzzles selbst:
Wir geben jeder inneren Kante eine zufällige Figur (auf dem einen Teil `z`, auf dem anderen `-z`), die Randseiten bekommen zufällige Figuren, mischen die Teile und drehen sie zufällig.
Dann ist das Puzzle bestimmt lösbar. Der Zufall ist mit `random.seed` festgelegt, damit immer dasselbe herauskommt.

In [ ]:
import random


def erzeuge_puzzle(reihen, figuren, startwert):
    random.seed(startwert)
    n = reihen * reihen
    seiten = []
    for p in range(n):
        seiten.append([0, 0, 0])
    for a, seite_a, b, seite_b in finde_kanten(n):
        z = random.randint(1, figuren)
        if random.randint(0, 1) == 0:
            z = -z
        seiten[a][seite_a] = z
        seiten[b][seite_b] = -z
    for teil in seiten:                           # Randseiten: noch leer, zufällig füllen
        for s in range(3):
            if teil[s] == 0:
                teil[s] = random.choice([-1, 1]) * random.randint(1, figuren)
    random.shuffle(seiten)
    for teil in seiten:
        d = random.randint(0, 2)
        teil[:] = teil[d:] + teil[:d]             # zyklisch drehen
    return seiten


for reihen, figuren in [(3, 4), (4, 5), (4, 8)]:
    teile = erzeuge_puzzle(reihen, figuren, 1)
    loesungen, aufrufe = loese(teile)
    print(f'{reihen * reihen} Teile, {figuren} Figuren: lösbar: {len(loesungen) == 1}, {aufrufe} Aufrufe, geprüft: {pruefe_loesung(teile, loesungen[0])}')

9 Teile, 4 Figuren: lösbar: True, 1436 Aufrufe, geprüft: True
16 Teile, 5 Figuren: lösbar: True, 37157 Aufrufe, geprüft: True
16 Teile, 8 Figuren: lösbar: True, 43261 Aufrufe, geprüft: True


Die Zahl der Aufrufe wächst schnell mit der Größe: Von 9 auf 16 Teile werden aus gut tausend Aufrufen rund vierzigtausend, und bei einem Versuch mit 25 Teilen waren es über 400 000.
Für die neun Teile der Aufgabe ist das Verfahren mehr als schnell genug.

## Zum Schluss: eine Lösung mit MiniZinc

Auch hier passt MiniZinc gut: Wir müssen nur beschreiben, **was** eine Lösung ist, und der Solver sucht sie.
Es gibt zwei Arten von Entscheidungen, nämlich welches Teil auf welcher Position liegt (eine Permutation, also `alldifferent`) und wie es gedreht ist. Dazu kommt **eine Bedingung je Kante**.
Das Modell enthält genau die Zutaten, die wir oben auch hatten: die Tabelle der Kanten, die Zahl `teil[t, (s + d) mod 3]` und die Summe 0.

Die MiniZinc-Blöcke unten sind **kein Python**: Kopiere sie in die MiniZinc IDE (oder in den
[MiniZinc Playground](https://play.minizinc.org/)) und führe sie dort aus.

#### Die Daten

Die Python-Zelle macht aus einer Beispieldatei den Inhalt der Datendatei (Kopiere die Ausgabe in eine Datei `puzzle3.dzn`).
Die Teile und Positionen zählen in MiniZinc ab 1, die Seiten ab 0, damit `mod 3` stimmt. `array2d(1..9, 0..2, [...])` füllt eine Tabelle zeilenweise: eine Zeile je Teil.
Die Kanten gehen als zweite Tabelle hinein (je Zeile: Position a, Seite von a, Position b, Seite von b).

In [ ]:
def als_dzn(dateiname):
    teile = lies_puzzle(dateiname)
    kanten = finde_kanten(len(teile))
    zahlen = []
    for teil in teile:
        zahlen = zahlen + teil
    kantenzahlen = []
    for a, seite_a, b, seite_b in kanten:
        kantenzahlen = kantenzahlen + [a + 1, seite_a, b + 1, seite_b]      # Positionen ab 1 zählen
    text = 'n = ' + str(len(teile)) + ';\n'
    text = text + 'k = ' + str(len(kanten)) + ';\n'
    text = text + 'teil = array2d(1..' + str(len(teile)) + ', 0..2, ' + str(zahlen) + ');\n'
    text = text + 'kante = array2d(1..' + str(len(kanten)) + ', 1..4, ' + str(kantenzahlen) + ');'
    return text


print(als_dzn('beispieldaten/puzzle3.txt'))

n = 9;
k = 9;
teil = array2d(1..9, 0..2, [10, 10, 4, 9, 8, -7, 10, -5, 10, -2, 10, 7, 6, 5, -2, 2, 3, -4, -6, -8, 10, -9, 10, 10, 10, -3, 2]);
kante = array2d(1..9, 1..4, [2, 0, 3, 2, 3, 1, 4, 2, 1, 1, 3, 0, 5, 0, 6, 2, 6, 1, 7, 2, 2, 1, 6, 0, 7, 0, 8, 2, 8, 1, 9, 2, 4, 1, 8, 0]);


#### Das Modell

`teil_auf[p]` ist die Nummer des Teils auf Position `p` und `dreh[p]` seine Drehung. Das sind Variablen, die als **Index** in die Tabelle `teil` dienen: `teil[teil_auf[a], ...]` ist die Zahl eines Teils,
das der Solver erst noch auswählt. In der `forall`-Bedingung steht für jede Kante dieselbe Prüfung wie in unserer Funktion `passt`.
Gesucht ist eine beliebige Lösung (`solve satisfy`).

```minizinc
include "alldifferent.mzn";

% ----- Daten (aus der .dzn-Datei) -----
int: n;                                  % Anzahl der Teile
int: k;                                  % Anzahl der Kanten
array[1..n, 0..2] of int: teil;          % teil[t, s]: Zahl auf Seite s von Teil t
array[1..k, 1..4] of int: kante;         % je Kante: Position a, Seite von a, Position b, Seite von b

% ----- Entscheidungsvariablen -----
array[1..n] of var 1..n: teil_auf;       % teil_auf[p]: Nummer des Teils auf Position p
array[1..n] of var 0..2: dreh;           % dreh[p]: Drehung des Teils auf Position p

% ----- Bedingungen -----
constraint alldifferent(teil_auf);       % jedes Teil genau einmal

% an jeder Kante müssen die Zahlen beider Teile zusammen 0 ergeben
constraint forall(e in 1..k)(
  teil[teil_auf[kante[e, 1]], (kante[e, 2] + dreh[kante[e, 1]]) mod 3]
  + teil[teil_auf[kante[e, 3]], (kante[e, 4] + dreh[kante[e, 3]]) mod 3] = 0
);

solve satisfy;

output ["Teile:    \(teil_auf)\n", "Drehung:  \(dreh)\n"];
```

Mit den Daten von `puzzle3.txt`:

**Ausgabe:**

```
Teile:    [8, 4, 2, 7, 1, 6, 9, 5, 3]
Drehung:  [2, 2, 0, 2, 2, 0, 2, 0, 2]
----------
```

Gelesen wird das so: Teil 8 liegt auf Position 1 (oben) mit Drehung 2, Teil 4 auf Position 2 mit Drehung 2, Teil 2 auf Position 3 mit Drehung 0 und so weiter.
Es ist eine der drei Lösungen von `puzzle3.txt` (siehe oben: Jede Lösung gibt es dreimal). Beim Erzeugen dieses Notebooks haben wir sie mit unserer Funktion `pruefe_loesung` kontrolliert (Teile ab 1, Drehungen wie oben).
Für `puzzle0.txt`, `puzzle1.txt` und `puzzle2.txt` findet der Solver (Gecode) ebenfalls jeweils in unter einer Sekunde eine Lösung, auch dort, wo es viele gibt.

Auf einem Puzzle ohne Lösung, wie unserem veränderten `puzzle3.txt`, meldet MiniZinc `=====UNSATISFIABLE=====`.

*Vergleich:* Der Solver ist hier nicht viel besser als unser eigenes Backtracking, das für neun Teile schon schnell genug ist. Der Vorteil liegt in der Kürze: Das Modell ist deutlich kürzer als unser Python-Programm.
Dafür braucht man die Kantentabelle, die in unserem Python-Programm auch schon da war.